In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import sys
print(sys.executable)

import os
os.getcwd()

import gc

In [ ]:
import numpy as np
from scipy.interpolate import interp1d
from scipy.integrate import odeint
from scipy.integrate import solve_ivp
import time
import random
import sys
from tqdm import tqdm
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append('/user/leuven/384/vsc38419/0scillons/February26Oscillons/Feb26')

from core.grid import Grid
from core.rhsevolution_MG import *
from core.spacing import *
from core.display import *
from core.statevector import *
from matter.scalarmatter_MG import *

from bssn.oscillondiagnostic import (get_oscillon_diagnostic,
                                     plot_density_profiles_at_times,
                                     plot_paper_diagnostics)

from initialdata.ModifiedGravityInitialConditions import *

from backgrounds.sphericalbackground import *
from bssn.constraintsdiagnostic import *
from bssn.ahfinder import *

In [ ]:
####################################################
# SWEEP PARAMETERS — lists of values to loop over
# (itertools.product handles unequal lengths)
####################################################

amplitudes   = [-2e-2]                  # perturbation amplitude
widths_R     = [3]                      # perturbation width / R
lambda_GBs   = [0, 0.001]              # GB coupling constant
mus          = [0.08]                   # self-interaction (key into TABLE_I)

####################################################
# FIXED PARAMETERS
####################################################

scalar_mu = 1
a = 0.2
b = 0.4
chi0 = 0.15
coupling = 'quadratic'

TABLE_I = {
    0.04: dict(phi=-6.03334e-2, dphi=2.20256e-2, H=1.55744e-2),
    0.05: dict(phi=-7.36454e-2, dphi=2.72499e-2, H=1.92686e-2),
    0.06: dict(phi=-8.64102e-2, dphi=3.23761e-2, H=2.28934e-2),
    0.07: dict(phi=-9.86792e-2, dphi=3.74094e-2, H=2.64525e-2),
    0.08: dict(phi=-1.10495e-1, dphi=4.23540e-2, H=2.99488e-2),
    0.09: dict(phi=-1.21893e-1, dphi=4.72136e-2, H=3.33851e-2),
    0.10: dict(phi=-1.32906e-1, dphi=5.19917e-2, H=3.67637e-2),
}

# Time
T = 800
num_points_t = 1000
dt = T / num_points_t
t = np.linspace(0, T - dt, num_points_t)

In [ ]:
####################################################
# RESOLUTION & GRID SETTINGS
####################################################

r_max = 150
min_dr = 1 / 4          # single resolution for the sweep
max_dr = 2

resolutions = {
    'LR': 1 / 4,
    'MR': 1 / 8,
    'HR': 1 / 16,
}

# Set to ['LR'] for quick sweeps, or ['LR','MR','HR'] for convergence
active_resolutions = ['LR']

print(f"Active resolutions: {active_resolutions}")
for tag, dr in resolutions.items():
    if tag in active_resolutions:
        print(f"  {tag}: min_dr = {dr}")

In [ ]:
####################################################
# DATA DIRECTORY — one subfolder per parameter set
####################################################
import itertools

NOTEBOOK_DIR = "/user/leuven/384/vsc38419/0scillons/February26Oscillons/Feb26/Notebooks"
BASE_DIR = os.path.join(NOTEBOOK_DIR, "oscillon_MG")

def make_run_dir(lgb, mu, a_param, b_param, amp, R, dr):
    """Build a unique subfolder name encoding every physics + grid parameter."""
    tag = (f"lgb{lgb}_mu{mu}_a{a_param}_b{b_param}"
           f"_amp{amp}_R{R}_dr{dr}")
    run_dir = os.path.join(BASE_DIR, tag)
    os.makedirs(run_dir, exist_ok=True)
    return run_dir

combos = list(itertools.product(amplitudes, widths_R, lambda_GBs, mus))
n_runs = len(combos) * len(active_resolutions)
print(f"{len(combos)} parameter combinations x {len(active_resolutions)} resolutions = {n_runs} total runs")
for i, (amp, wR, lgb, mu) in enumerate(combos):
    print(f"  [{i}] amp={amp}, R={wR}, lgb={lgb}, mu={mu}")

In [ ]:
####################################################
# MAIN SIMULATION LOOP
####################################################

run_counter = 0

for amp, wR, lgb, mu in combos:

    u_val = TABLE_I[mu]['phi']
    v_val = TABLE_I[mu]['dphi']

    my_matter = ScalarMatter(scalar_mu, mu)
    my_state_vector = StateVector(my_matter)

    for res_tag in active_resolutions:
        run_counter += 1
        min_dr_cur = resolutions[res_tag]

        spacing_cur = CubicSpacing(**CubicSpacing.get_parameters(r_max, min_dr_cur, max_dr))
        grid_cur = Grid(spacing_cur, my_state_vector)
        r_cur = grid_cur.r
        background_cur = FlatSphericalBackground(r_cur)

        run_dir = make_run_dir(lgb, mu, a, b, amp, wR, min_dr_cur)

        print(f"\n{'='*60}")
        print(f"Run {run_counter}/{n_runs}  [{res_tag}]")
        print(f"  lgb={lgb}, mu={mu}, a={a}, b={b}, amp={amp}, R={wR}, dr={min_dr_cur}")
        print(f"  Grid: {r_cur.size} points  ->  {run_dir}")

        initial_state = get_initial_state(
            grid_cur, background_cur, (lgb, a, b, chi0, coupling),
            my_matter, amp, wR, scalar_mu, u_val, v_val)

        with tqdm(total=1000, unit="\u2030") as progress_bar:
            dense_sol = solve_ivp(
                get_rhs, [0, T], initial_state,
                args=(grid_cur, background_cur, my_matter, progress_bar,
                      [0, T/1000], a, b, lgb, coupling),
                max_step=0.4 * min_dr_cur,
                method='RK45',
                dense_output=True)

        solution = dense_sol.sol(t).T

        np.save(os.path.join(run_dir, "solution.npy"), solution)
        np.save(os.path.join(run_dir, "t.npy"), t)
        np.save(os.path.join(run_dir, "r.npy"), r_cur)
        print(f"  Saved: shape {solution.shape}")

        del dense_sol, solution, initial_state
        gc.collect()

print(f"\nAll {run_counter} runs complete.")

In [ ]:
####################################################
# DIAGNOSTICS — load every saved run
####################################################

from bssn.oscillondiagnostic import (
    get_oscillon_diagnostic,
    plot_paper_diagnostics,
    plot_density_profiles_at_times,
    plot_density_contrast_profiles,
    plot_density_contrast_vs_lna,
    plot_central_density_vs_lna,
    plot_density_contrast_comparison,
    plot_hubble_vs_lna,
)

r_max_diag = 100.0
all_diagnostics = {}

for amp, wR, lgb, mu in combos:
    my_matter_diag = ScalarMatter(scalar_mu, mu)
    my_sv_diag = StateVector(my_matter_diag)

    for res_tag in active_resolutions:
        min_dr_cur = resolutions[res_tag]
        run_dir = make_run_dir(lgb, mu, a, b, amp, wR, min_dr_cur)

        sol = np.load(os.path.join(run_dir, "solution.npy"))
        t_loaded = np.load(os.path.join(run_dir, "t.npy"))

        spacing_d = CubicSpacing(**CubicSpacing.get_parameters(r_max, min_dr_cur, max_dr))
        grid_d = Grid(spacing_d, my_sv_diag)
        bg_d = FlatSphericalBackground(grid_d.r)

        params_d = (lgb, a, b, chi0, coupling)
        label = f"lgb={lgb}, mu={mu}, amp={amp}, R={wR}, {res_tag}"
        print(f"Diagnosing: {label}  (shape {sol.shape})")

        osc = get_oscillon_diagnostic(sol, t_loaded, grid_d, bg_d,
                                       my_matter_diag, params_d,
                                       surface_threshold=0.05,
                                       r_max_diag=r_max_diag)

        key = (amp, wR, lgb, mu, res_tag)
        all_diagnostics[key] = dict(osc=osc, t=t_loaded, grid=grid_d,
                                    bg=bg_d, matter=my_matter_diag,
                                    params=params_d, label=label,
                                    run_dir=run_dir)

        del sol; gc.collect()

print(f"\nLoaded {len(all_diagnostics)} diagnostic sets.")
print("Keys stored per run:", list(list(all_diagnostics.values())[0]['osc'].keys()))

In [ ]:
####################################################
# Paper-style: delta_c and rho_c vs ln(a) — all runs
####################################################

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(9, 9), sharex=True)

for key, entry in all_diagnostics.items():
    osc = entry['osc']
    lbl = entry['label']

    ln_a    = osc["ln_a"]
    delta_c = osc["delta_c"]
    rho_c   = osc["rho_c"]
    rho_bar = osc["rho_bar"]

    pos_dc = delta_c > 0
    log_dc = np.full_like(delta_c, np.nan)
    log_dc[pos_dc] = np.log10(delta_c[pos_dc])

    pos_rc = rho_c > 0
    log_rc = np.full_like(rho_c, np.nan)
    log_rc[pos_rc] = np.log10(rho_c[pos_rc])

    pos_rb = rho_bar > 0
    log_rb = np.full_like(rho_bar, np.nan)
    log_rb[pos_rb] = np.log10(rho_bar[pos_rb])

    ax_top.plot(ln_a, log_dc, lw=1.5, label=lbl)
    ax_bot.plot(ln_a, log_rc, lw=1.5, label=rf'$\rho_c$ {lbl}')
    ax_bot.plot(ln_a, log_rb, '--', lw=1.0, alpha=0.5, label=rf'$\bar{{\rho}}$ {lbl}')

ax_top.set_ylabel(r'$\log_{10}(\delta_c)$', fontsize=13)
ax_top.set_title(r'Density contrast  $\delta_c \equiv \rho_c/\bar\rho - 1$', fontsize=13)
ax_top.legend(fontsize=8, loc='upper left')
ax_top.grid(True, alpha=0.3)

ax_bot.set_xlabel(r'$\ln(a)$', fontsize=13)
ax_bot.set_ylabel(r'$\log_{10}(\rho)$', fontsize=13)
ax_bot.set_title(r'Central and background densities', fontsize=13)
ax_bot.legend(fontsize=8, loc='upper right')
ax_bot.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
####################################################
# CONVERGENCE ORDER (requires LR, MR, HR active)
####################################################

if not ({'LR', 'MR', 'HR'}.issubset(active_resolutions)):
    print("Skipping convergence — need all three resolutions active.")
else:
    diag_keys = {
        'delta_c': r'$\delta_c$',
        'rho_c':   r'$\rho_c$',
        'rho_bar': r'$\bar\rho$',
        'M':       r'$M$',
        'R_osc':   r'$R_{\rm osc}$',
        'C':       r'$\mathcal{C}$',
    }

    for amp, wR, lgb, mu in combos:
        k_LR = (amp, wR, lgb, mu, 'LR')
        k_MR = (amp, wR, lgb, mu, 'MR')
        k_HR = (amp, wR, lgb, mu, 'HR')

        osc_LR = all_diagnostics[k_LR]['osc']
        osc_MR = all_diagnostics[k_MR]['osc']
        osc_HR = all_diagnostics[k_HR]['osc']

        fig, axes = plt.subplots(3, 2, figsize=(14, 14))
        axes = axes.ravel()

        for ax_idx, (key, label) in enumerate(diag_keys.items()):
            diff_LM = np.abs(osc_LR[key] - osc_MR[key])
            diff_MH = np.abs(osc_MR[key] - osc_HR[key])

            with np.errstate(divide='ignore', invalid='ignore'):
                order = np.log2(diff_LM / diff_MH)

            valid = (diff_MH > 1e-14) & np.isfinite(order)

            ax = axes[ax_idx]
            ax.plot(osc_LR['ln_a'][valid], order[valid], 'k-', lw=1.0)
            ax.axhline(4, color='r', ls='--', lw=0.8, label='4th order')
            ax.axhline(2, color='g', ls='--', lw=0.8, label='2nd order')
            ax.set_xlabel('ln(a)')
            ax.set_ylabel('Convergence order p')
            ax.set_title(label)
            ax.set_ylim(-1, 8)
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)

        fig.suptitle(rf'Convergence:  lgb={lgb}, mu={mu}, amp={amp}, R={wR}',
                     fontsize=13)
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

In [ ]:
####################################################
# Spatial convergence of rho(r) (requires LR, MR, HR)
####################################################

if not ({'LR', 'MR', 'HR'}.issubset(active_resolutions)):
    print("Skipping spatial convergence — need all three resolutions active.")
else:
    t_snapshot = 600.0

    for amp, wR, lgb, mu in combos:
        k_LR = (amp, wR, lgb, mu, 'LR')
        k_MR = (amp, wR, lgb, mu, 'MR')
        k_HR = (amp, wR, lgb, mu, 'HR')

        osc_LR = all_diagnostics[k_LR]['osc']
        osc_MR = all_diagnostics[k_MR]['osc']
        osc_HR = all_diagnostics[k_HR]['osc']

        r_LR = all_diagnostics[k_LR]['grid'].r
        r_MR = all_diagnostics[k_MR]['grid'].r
        r_HR = all_diagnostics[k_HR]['grid'].r

        t_arr = all_diagnostics[k_HR]['t']
        idx_snap = np.argmin(np.abs(t_arr - t_snapshot))
        print(f"Snapshot at t = {t_arr[idx_snap]:.1f} — lgb={lgb}, mu={mu}, amp={amp}, R={wR}")

        rho_LR_on_HR = interp1d(r_LR, osc_LR["rho"][idx_snap], kind='cubic',
                                 fill_value='extrapolate')(r_HR)
        rho_MR_on_HR = interp1d(r_MR, osc_MR["rho"][idx_snap], kind='cubic',
                                 fill_value='extrapolate')(r_HR)
        rho_HR_on_HR = osc_HR["rho"][idx_snap]

        diff_ML = np.abs(rho_MR_on_HR - rho_LR_on_HR)
        diff_HM = np.abs(rho_HR_on_HR - rho_MR_on_HR)

        r_plot_max = 30.0
        pmask = (r_HR > 0.5) & (r_HR < r_plot_max)

        fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(8, 7), sharex=True,
                                              gridspec_kw={'height_ratios': [1, 1.3]})

        ax_top.plot(r_HR[pmask], rho_HR_on_HR[pmask], 'k-', lw=1.5)
        ax_top.set_ylabel(r'$\rho \;\;[m^2 M_{\rm Pl}^2]$', fontsize=12)
        ax_top.set_title(rf'$\rho(r)$ at $t={t_arr[idx_snap]:.0f}$ — lgb={lgb}, mu={mu}',
                         fontsize=13)
        ax_top.ticklabel_format(axis='y', style='sci', scilimits=(-4,-4))
        ax_top.grid(True, alpha=0.3)

        floor = 1e-30
        log_diff_ML = np.log10(np.maximum(diff_ML[pmask], floor))

        log_HM_2nd = np.log10(np.maximum(4.0 * diff_HM[pmask], floor))
        log_HM_4th = np.log10(np.maximum(16.0 * diff_HM[pmask], floor))

        ax_bot.plot(r_HR[pmask], log_diff_ML, 'b-', lw=1.2,
                    label=r'$|\rho_{MR} - \rho_{LR}|$')
        ax_bot.plot(r_HR[pmask], log_HM_2nd, 'r--', lw=1.0, alpha=0.7,
                    label=r'$4\times|\rho_{HR}-\rho_{MR}|$ (2nd)')
        ax_bot.plot(r_HR[pmask], log_HM_4th, 'g--', lw=1.0, alpha=0.7,
                    label=r'$16\times|\rho_{HR}-\rho_{MR}|$ (4th)')
        ax_bot.set_xlabel(r'$r / m^{-1}$', fontsize=12)
        ax_bot.set_ylabel(r'$\log_{10}(\Delta\rho)$', fontsize=12)
        ax_bot.legend(fontsize=9)
        ax_bot.grid(True, alpha=0.3)

        fig.tight_layout()
        plt.show()